In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

import pickle

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

In [2]:
df = pd.read_csv("recommender_model_dataset.csv")

print(df.shape)
display(df.head())
print(df.columns.tolist())

(5643, 47)


,player_id,season,alter,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,...,position_Linksaußen,position_Mittelfeld,position_Mittelstürmer,position_Offensives Mittelfeld,position_Rechter Verteidiger,position_Rechtes Mittelfeld,position_Rechtsaußen,position_Sturm,position_Torwart,position_Zentrales Mittelfeld
0,2866,2020,37.0,0.000000,0,0.000000,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,3391,2020,34.0,0.333333,1,0.000000,0,0.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
2,4779,2021,39.0,0.083333,2,0.041667,1,0.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
3,10058,2020,35.0,0.000000,0,0.000000,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,10058,2021,36.0,0.000000,0,0.000000,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


['player_id', 'season', 'alter', 'tore_pro_spiel', 'tore_abs', 'assists_pro_spiel', 'assists_abs', 'rote_karten_pro_spiel', 'gelbe_karten_pro_spiel', 'gelb_rote_karten_pro_spiel', 'rote_karten_abs', 'gelbe_karten_abs', 'gelb_rote_karten_abs', 'startelf_pro_spiel', 'startelf_abs', 'minuten_pro_spiel', 'minuten_abs', 'anzahl_spiele', 'team_tore_pro_spiel', 'team_tore_abs', 'team_gegentore_pro_spiel', 'team_gegentore_abs', 'siege_pro_spiel', 'unentschieden_pro_spiel', 'niederlagen_pro_spiel', 'siege_abs', 'unentschieden_abs', 'niederlagen_abs', 'prozent_1_liga', 'prozent_pl', 'ziel_rating_avg_naechste_saison', 'position_Abwehr', 'position_Defensives Mittelfeld', 'position_Hängende Spitze', 'position_Innenverteidiger', 'position_Linker Verteidiger', 'position_Linkes Mittelfeld', 'position_Linksaußen', 'position_Mittelfeld', 'position_Mittelstürmer', 'position_Offensives Mittelfeld', 'position_Rechter Verteidiger', 'position_Rechtes Mittelfeld', 'position_Rechtsaußen', 'position_Sturm', 'po

In [3]:
print("Saisons:")
print(df["season"].value_counts().sort_index())

print("\nFehlende Werte:")
display(df.isna().mean().sort_values(ascending=False).head(20))

print("\nTarget Summary:")
print(df["ziel_rating_avg_naechste_saison"].describe())

Saisons:
season
2020     987
2021    1047
2022    1214
2023    1245
2024    1150
Name: count, dtype: int64

Fehlende Werte:


alter                              0.000709
player_id                          0.000000
position_Linker Verteidiger        0.000000
unentschieden_abs                  0.000000
niederlagen_abs                    0.000000
prozent_1_liga                     0.000000
prozent_pl                         0.000000
ziel_rating_avg_naechste_saison    0.000000
position_Abwehr                    0.000000
position_Defensives Mittelfeld     0.000000
position_Hängende Spitze           0.000000
position_Innenverteidiger          0.000000
position_Linkes Mittelfeld         0.000000
niederlagen_pro_spiel              0.000000
position_Linksaußen                0.000000
position_Mittelfeld                0.000000
position_Mittelstürmer             0.000000
position_Offensives Mittelfeld     0.000000
position_Rechter Verteidiger       0.000000
position_Rechtes Mittelfeld        0.000000
dtype: float64


Target Summary:
count    5643.000000
mean        6.846437
std         0.229117
min         3.700000
25%         6.700000
50%         6.832143
75%         6.976000
max         8.400000
Name: ziel_rating_avg_naechste_saison, dtype: float64


In [4]:
target_col = "ziel_rating_avg_naechste_saison"

drop_cols = [
    target_col,
    # player_id nicht als Feature verwenden
    "player_id"
]

X = df.drop(columns=drop_cols).copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

X shape: (5643, 45)
y shape: (5643,)


,season,alter,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,rote_karten_abs,...,position_Linksaußen,position_Mittelfeld,position_Mittelstürmer,position_Offensives Mittelfeld,position_Rechter Verteidiger,position_Rechtes Mittelfeld,position_Rechtsaußen,position_Sturm,position_Torwart,position_Zentrales Mittelfeld
0,2020,37.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2020,34.0,0.333333,1,0.000000,0,0.0,0.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
2,2021,39.0,0.083333,2,0.041667,1,0.0,0.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
3,2020,35.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,2021,36.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
train_mask = df["season"] <= 2023
test_mask = df["season"] == 2024

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

season_train = df.loc[train_mask, "season"].copy()
season_test = df.loc[test_mask, "season"].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain seasons:")
print(season_train.value_counts().sort_index())

print("\nTest seasons:")
print(season_test.value_counts().sort_index())

Train shape: (4493, 45)
Test shape: (1150, 45)

Train seasons:
season
2020     987
2021    1047
2022    1214
2023    1245
Name: count, dtype: int64

Test seasons:
season
2024    1150
Name: count, dtype: int64


In [6]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = root_mean_squared_error(y_test, baseline_preds)
baseline_r2 = r2_score(y_test, baseline_preds)

print("Baseline Performance")
print(f"MAE :  {baseline_mae:.4f}")
print(f"RMSE:  {baseline_rmse:.4f}")
print(f"R²  :  {baseline_r2:.4f}")

Baseline Performance
MAE :  0.1784
RMSE:  0.2351
R²  :  -0.0003


In [7]:
unique_train_seasons = sorted(season_train.unique())
print("Train seasons for CV:", unique_train_seasons)

cv = GroupKFold(n_splits=len(unique_train_seasons))
groups = season_train

Train seasons for CV: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [8]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

In [9]:
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [2, 3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 2, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0, 5.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0]
}

In [10]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state=42,
    refit=True,
    return_train_score=True
)

random_search.fit(X_train, y_train, groups=groups)

Fitting 4 folds for each of 40 candidates, totalling 160 fits
[CV] END colsample_bytree=1.0, gamma=0.5, learning_rate=0.1, max_depth=8, min_child_weight=2, n_estimators=300, reg_alpha=1.0, reg_lambda=1.0, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=1.0, gamma=0.5, learning_rate=0.1, max_depth=8, min_child_weight=2, n_estimators=300, reg_alpha=1.0, reg_lambda=1.0, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=1.0, gamma=0.5, learning_rate=0.1, max_depth=4, min_child_weight=1, n_estimators=300, reg_alpha=5.0, reg_lambda=2.0, subsample=0.6; total time=   0.1s
[CV] END colsample_bytree=1.0, gamma=0.5, learning_rate=0.1, max_depth=8, min_child_weight=2, n_estimators=300, reg_alpha=1.0, reg_lambda=1.0, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=1.0, gamma=0.5, learning_rate=0.1, max_depth=4, min_child_weight=1, n_estimators=300, reg_alpha=5.0, reg_lambda=2.0, subsample=0.6; total time=   0.1s
[CV] END colsample_bytree=1.0, gamma=0.5, learning_ra

,estimator,"XGBRegressor(...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.6, 0.7, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [2, 3, ...], ...}"
,n_iter,40
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,GroupKFold(n_...shuffle=False)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [11]:
print("Best score (CV, neg MAE):", random_search.best_score_)
print("Best params:")
print(random_search.best_params_)

Best score (CV, neg MAE): -0.13914239673642897
Best params:
{'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 5.0, 'n_estimators': 100, 'min_child_weight': 2, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}


In [12]:
cv_results = pd.DataFrame(random_search.cv_results_)
cv_results = cv_results.sort_values("rank_test_score")

cols_to_show = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "param_n_estimators",
    "param_max_depth",
    "param_learning_rate",
    "param_subsample",
    "param_colsample_bytree",
    "param_min_child_weight",
    "param_gamma",
    "param_reg_alpha",
    "param_reg_lambda",
]

display(cv_results[cols_to_show].head(10))

,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_n_estimators,param_max_depth,param_learning_rate,param_subsample,param_colsample_bytree,param_min_child_weight,param_gamma,param_reg_alpha,param_reg_lambda
30,1,-0.139142,0.004397,-0.133804,100,4,0.05,0.7,1.0,2,0.0,5.00,2.0
35,2,-0.139193,0.004830,-0.129832,100,5,0.10,0.6,0.6,1,0.0,5.00,0.5
38,3,-0.139470,0.004683,-0.131943,800,3,0.15,1.0,1.0,2,0.0,5.00,0.5
11,4,-0.140068,0.004682,-0.132329,100,8,0.15,0.8,0.6,1,0.3,0.01,5.0
10,5,-0.140072,0.004674,-0.137059,800,2,0.03,0.7,0.8,5,0.5,0.01,2.0
8,6,-0.140189,0.004496,-0.135138,500,5,0.01,1.0,0.8,3,0.3,0.10,10.0
3,7,-0.140231,0.005391,-0.118474,300,8,0.01,0.9,0.6,2,0.1,0.00,5.0
4,8,-0.140294,0.005600,-0.134351,300,2,0.15,0.6,0.7,2,0.3,0.10,1.0
32,9,-0.140384,0.005979,-0.130765,200,2,0.10,0.6,0.7,5,0.0,0.01,2.0
39,10,-0.140465,0.005426,-0.126834,100,3,0.10,0.9,0.7,1,0.0,0.10,0.5


In [13]:
best_model = random_search.best_estimator_

test_preds = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = root_mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)

print("XGBoost Test Performance")
print(f"MAE :  {test_mae:.4f}")
print(f"RMSE:  {test_rmse:.4f}")
print(f"R²  :  {test_r2:.4f}")

XGBoost Test Performance
MAE :  0.1516
RMSE:  0.2060
R²  :  0.2315


In [14]:
comparison = pd.DataFrame({
    "Modell": ["Baseline", "XGBoost"],
    "MAE": [baseline_mae, test_mae],
    "RMSE": [baseline_rmse, test_rmse],
    "R2": [baseline_r2, test_r2]
})

display(comparison)

,Modell,MAE,RMSE,R2
0,Baseline,0.178370,0.235065,-0.000298
1,XGBoost,0.151574,0.206032,0.231542


In [15]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(20))

,feature,importance
14,minuten_pro_spiel,0.197248
43,position_Torwart,0.152365
3,tore_abs,0.057111
32,position_Innenverteidiger,0.050410
18,team_tore_abs,0.045042
2,tore_pro_spiel,0.043499
17,team_tore_pro_spiel,0.042001
39,position_Rechter Verteidiger,0.037872
37,position_Mittelstürmer,0.031186
44,position_Zentrales Mittelfeld,0.029905


In [16]:
pred_df = df.loc[test_mask, ["player_id", "season"]].copy()
pred_df["y_true"] = y_test.values
pred_df["y_pred"] = test_preds
pred_df["abs_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])

display(pred_df.sort_values("abs_error").head(20))
display(pred_df.sort_values("abs_error", ascending=False).head(20))

,player_id,season,y_true,y_pred,abs_error
3026,539811,2024,6.900000,6.900270,0.000270
4472,805762,2024,6.711765,6.712059,0.000294
2640,506884,2024,6.850000,6.850385,0.000385
748,213160,2024,6.938462,6.938065,0.000397
4906,921613,2024,6.847059,6.847488,0.000430
31,37522,2024,6.982353,6.981684,0.000669
3485,620765,2024,6.733333,6.732592,0.000741
5143,951660,2024,7.108333,7.107553,0.000780
3706,645533,2024,6.961538,6.960444,0.001094
1120,284804,2024,6.800000,6.801208,0.001208


,player_id,season,y_true,y_pred,abs_error
5095,932195,2024,8.100000,7.071133,1.028867
4223,708381,2024,5.850000,6.842986,0.992986
1000,260163,2024,7.915789,6.988348,0.927442
1369,322512,2024,7.673684,6.842659,0.831025
4748,897461,2024,7.800000,7.004052,0.795948
4626,843728,2024,7.543750,6.802610,0.741140
5553,1206410,2024,7.700000,6.970294,0.729706
4978,925736,2024,5.966667,6.692831,0.726164
4916,922291,2024,7.585714,6.863089,0.722625
5144,954029,2024,7.566667,6.844640,0.722026


In [17]:
with open("recommender_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Saved as: recommender_model.pkl")

Saved as: recommender_model.pkl


In [18]:
pred_df = df.loc[test_mask, ["player_id", "season"]].copy()
pred_df["y_true"] = y_test.values
pred_df["y_pred"] = test_preds
pred_df["abs_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])